# SIREN Segmentation Training Pipeline

**셀 순서**: 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8  
**재시작 후**: Cell 1은 반드시 재실행. Cell 3 이후는 체크포인트가 있으면 스킵 가능.

| Cell | 역할 | 체크포인트 |
|------|------|----------|
| 1 | 환경설정 (설치·Drive·clone·symlink) | 없음 (항상 실행) |
| 2 | sys.path + 경로 초기화 | 없음 (항상 실행) |
| 3 | 데이터 로드 + 정규화 | `processed/normalized_rows.pkl` |
| 4 | 도메인 필터 + 온톨로지 테이블 | `processed/surface_ontology.pkl` |
| 5 | Split 레코드 + 샘플링 매니페스트 | `processed/split_manifest.pkl` |
| 6 | 압축 해제 + Export | `curated_local/` 디렉터리 존재 여부 |
| 7 | 학습 전 데이터셋 검증 | 없음 |
| 8 | YOLOv8n-seg 학습 | `runs/` 결과 + Drive 미러 |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 1 — 환경 설정 (항상 재실행)                                ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, sys, subprocess
from pathlib import Path

# ── 1-A. 패키지 설치 ────────────────────────────────────────────────
print("[1-A] 패키지 설치 중...")
subprocess.run(["pip", "install", "-q", "ultralytics", "pyyaml"], check=True)
print("[1-A] 완료")

# ── 1-B. Google Drive 마운트 ────────────────────────────────────────
print("[1-B] Drive 마운트 중...")
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
assert (DRIVE_ROOT / "siren_repo/data/analysis/dataset_index.csv").exists(), \
    f"dataset_index.csv not found in {DRIVE_ROOT}/siren_repo/data/analysis/"
print("[1-B] Drive 마운트 완료")

# ── 1-C. repo clone / 최신화 ────────────────────────────────────────
REPO_PATH = Path("/content/siren")
if not (REPO_PATH / ".git").exists():
    print("[1-C] git clone 중...")
    subprocess.run(
        ["git", "clone", "https://github.com/zldnlto/siren-api.git", str(REPO_PATH)],
        check=True
    )
else:
    print("[1-C] git pull 중...")
    subprocess.run(["git", "-C", str(REPO_PATH), "pull", "--ff-only"], check=True)
print("[1-C] 완료")

# ── 1-D. vision/data symlink ────────────────────────────────────────
VISION_DATA = REPO_PATH / "vision" / "data"
DRIVE_DATA  = DRIVE_ROOT / "siren_repo" / "data"

if VISION_DATA.is_symlink() or VISION_DATA.exists():
    if VISION_DATA.is_symlink():
        VISION_DATA.unlink()
    else:
        # 실제 디렉터리가 있으면 덮어쓰지 않음 — 이미 올바른 환경일 수 있음
        pass

if not VISION_DATA.exists():
    os.symlink(str(DRIVE_DATA), str(VISION_DATA))
    print(f"[1-D] symlink 생성: {VISION_DATA} → {DRIVE_DATA}")
else:
    print(f"[1-D] vision/data 이미 존재 (symlink 불필요)")

# ── 1 검증 ──────────────────────────────────────────────────────────
DATASET_INDEX_PATH = VISION_DATA / "analysis" / "dataset_index.csv"
assert DATASET_INDEX_PATH.exists(), \
    f"[오류] dataset_index.csv 없음: {DATASET_INDEX_PATH}\n"\
    "→ Drive 마운트 및 symlink 경로를 확인하세요."

SAMPLING_COLAB_CFG = REPO_PATH / "vision" / "configs" / "sampling_colab.yaml"
assert SAMPLING_COLAB_CFG.exists(), \
    f"[오류] sampling_colab.yaml 없음: {SAMPLING_COLAB_CFG}"

print("\n✅ Cell 1 완료")
print(f"   REPO_PATH       : {REPO_PATH}")
print(f"   DATASET_INDEX   : {DATASET_INDEX_PATH}")
print(f"   SAMPLING_CONFIG : {SAMPLING_COLAB_CFG}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 2 — sys.path + 경로 초기화 (항상 재실행)                   ║
# ╚══════════════════════════════════════════════════════════════════╝
import sys
from pathlib import Path

# ── 2-A. sys.path ───────────────────────────────────────────────────
REPO_PATH = Path("/content/siren")
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

# ── 2-B. 경로 상수 ──────────────────────────────────────────────────
from vision.src.settings import build_default_runtime_config, build_segmentation_runtime_config
from vision.src.data.config import (
    DEFAULT_SEGMENTATION_EXPORT_ROOT,
    DEFAULT_LABEL_MAP_ROOT,
)

_default_runtime = build_default_runtime_config()
paths = _default_runtime.paths

# Colab 실행 중 생성될 로컬 경로들
# export_segmentation_dataset output_root = 학습에 직접 사용할 curated 루트
CURATED_ROOT       = REPO_PATH / "vision" / "data" / "curated_local"
SAMPLED_IMAGES_DIR = Path("/content/sampled_images")   # sampled_surface.zip 추출 위치
LABELS_FLAT_DIR    = Path("/content/labels_flat")       # labels.zip 추출 위치
PROCESSED_DIR      = REPO_PATH / "vision" / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Drive 경로
DRIVE_ROOT         = Path("/content/drive/MyDrive")
DRIVE_DATA         = DRIVE_ROOT / "siren_repo" / "data"
DRIVE_RUNS         = DRIVE_ROOT / "siren" / "runs"
DRIVE_ZIPS         = DRIVE_DATA  # sampled_surface.zip이 있는 위치

# 체크포인트 파일 경로 (세션 내 캐싱)
CKPT_NORMALIZED    = PROCESSED_DIR / "normalized_rows.pkl"
CKPT_ONTOLOGY      = PROCESSED_DIR / "surface_ontology.pkl"
CKPT_SPLIT_RECORDS = PROCESSED_DIR / "split_records.pkl"
CKPT_MANIFEST      = PROCESSED_DIR / "split_manifest.pkl"

SAMPLING_COLAB_CFG = REPO_PATH / "vision" / "configs" / "sampling_colab.yaml"
TARGET_DOMAIN      = "표면처리"

# ── 2 검증 ──────────────────────────────────────────────────────────
from vision.src.data import load_dataset_index_rows
# 모듈 import만 검증 (실제 로드는 Cell 3)
assert callable(load_dataset_index_rows)

assert SAMPLING_COLAB_CFG.exists(), \
    f"[오류] sampling_colab.yaml 없음 — Cell 1을 먼저 실행하세요."

# canonical_class_name이 slugs를 받았을 때 영문을 반환하는지 확인
from vision.src.data.ontology import canonical_class_name, load_ontology_slugs
_slugs = load_ontology_slugs()
_test  = canonical_class_name("도막떨어짐", "도장", slugs=_slugs)
assert _test == "coating_drop_paint", \
    f"[오류] canonical_class_name 버그 미패치 — 반환값: {_test!r}\n"\
    "git pull 후 Colab 런타임을 재시작하세요."

print("✅ Cell 2 완료")
print(f"   REPO_PATH    : {REPO_PATH}")
print(f"   CURATED_ROOT : {CURATED_ROOT}")
print(f"   PROCESSED_DIR: {PROCESSED_DIR}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 3 — 데이터 로드 + 정규화                                   ║
# ╚══════════════════════════════════════════════════════════════════╝
import pickle
from vision.src.data import load_dataset_index_rows, normalize_rows

if CKPT_NORMALIZED.exists():
    print(f"[3] 체크포인트 로드: {CKPT_NORMALIZED}")
    with open(CKPT_NORMALIZED, "rb") as f:
        normalized = pickle.load(f)
else:
    print("[3] dataset_index.csv 로드 중...")
    raw_rows = load_dataset_index_rows()
    print(f"[3] {len(raw_rows):,}행 로드 완료. 정규화 중...")
    normalized = normalize_rows(raw_rows)
    with open(CKPT_NORMALIZED, "wb") as f:
        pickle.dump(normalized, f)
    print(f"[3] 체크포인트 저장: {CKPT_NORMALIZED}")

# ── 3 검증 ──────────────────────────────────────────────────────────
assert len(normalized) > 0, "[오류] normalize_rows 결과가 비어있음"

# canonical_class_name이 모두 영문 slug인지 확인
_korean_rows = [
    r for r in normalized[:1000]
    if any(ord(c) > 127 for c in r.canonical_class_name)
]
assert len(_korean_rows) == 0, \
    f"[오류] canonical_class_name에 한국어 포함: {_korean_rows[0].canonical_class_name!r}\n"\
    "→ git pull 후 런타임 재시작 필요."

# ontology_id 예시 출력
_sample = normalized[0]
print(f"\n✅ Cell 3 완료")
print(f"   총 행 수               : {len(normalized):,}")
print(f"   샘플 canonical_class   : {_sample.canonical_class_name}")
print(f"   샘플 ontology_id       : {_sample.ontology_id}")
print(f"   샘플 annotation_domain : {_sample.annotation_domain}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 4 — 도메인 필터 + 온톨로지 테이블                          ║
# ╚══════════════════════════════════════════════════════════════════╝
import pickle
from vision.src.data import build_ontology_table
from vision.src.data.ontology import load_ontology_slugs

if CKPT_ONTOLOGY.exists():
    print(f"[4] 체크포인트 로드: {CKPT_ONTOLOGY}")
    with open(CKPT_ONTOLOGY, "rb") as f:
        surface_ontology = pickle.load(f)
    surface_rows = [r for r in normalized if r.annotation_domain == TARGET_DOMAIN]
else:
    print(f"[4] {TARGET_DOMAIN} 도메인 필터링 중...")
    surface_rows = [r for r in normalized if r.annotation_domain == TARGET_DOMAIN]
    assert len(surface_rows) > 0, \
        f"[오류] domain={TARGET_DOMAIN!r} 행 없음. TARGET_DOMAIN 또는 annotation_domain 확인 필요."
    print(f"[4] 표면처리 행: {len(surface_rows):,} / 전체: {len(normalized):,}")

    slugs = load_ontology_slugs()
    surface_ontology = build_ontology_table(surface_rows, slugs=slugs)
    with open(CKPT_ONTOLOGY, "wb") as f:
        pickle.dump(surface_ontology, f)
    print(f"[4] 체크포인트 저장: {CKPT_ONTOLOGY}")

# ── 4 검증 ──────────────────────────────────────────────────────────
assert len(surface_ontology) > 0, "[오류] 온톨로지 테이블이 비어있음"

_korean_canonical = [
    r.canonical_class_name for r in surface_ontology
    if any(ord(c) > 127 for c in r.canonical_class_name)
]
assert len(_korean_canonical) == 0, \
    f"[오류] OntologyRecord에 한국어 canonical_class_name 있음: {_korean_canonical}\n"\
    "→ git pull 후 런타임 재시작, CKPT_ONTOLOGY 파일 삭제 후 재실행."

# segment 태스크를 허용하는 레코드가 있는지 확인
_seg_records = [r for r in surface_ontology if "segment" in r.allowed_task_types]
assert len(_seg_records) > 0, \
    "[오류] allowed_task_types에 'segment'를 포함한 레코드 없음"

print("\n✅ Cell 4 완료")
print(f"   온톨로지 레코드 수       : {len(surface_ontology)}")
print(f"   segment 가능 레코드 수   : {len(_seg_records)}")
for rec in surface_ontology:
    print(f"   {rec.canonical_class_name:40s} | bucket={rec.support_bucket}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 5 — Split 레코드 + 샘플링 매니페스트                       ║
# ╚══════════════════════════════════════════════════════════════════╝
import pickle
from vision.src.data import (
    build_image_split_records,
    build_sampling_manifest,
    save_sampling_manifest,
)
from vision.src.data.split_sampling import load_sampling_config

sampling_config = load_sampling_config(SAMPLING_COLAB_CFG)
print(f"[5] sampling config: {SAMPLING_COLAB_CFG.name}")
print(f"    train caps = {sampling_config.train_caps}")

if CKPT_MANIFEST.exists():
    print(f"[5] 체크포인트 로드: {CKPT_MANIFEST}")
    with open(CKPT_MANIFEST, "rb") as f:
        split_records, sampling_records, sampling_report = pickle.load(f)
else:
    print("[5] Split 레코드 생성 중...")
    split_records, leaks = build_image_split_records(
        surface_rows, ontology_records=surface_ontology
    )
    print(f"[5] split 레코드: {len(split_records):,}  split 오염 의심: {len(leaks)}")

    print("[5] 샘플링 매니페스트 생성 중...")
    sampling_records, sampling_report = build_sampling_manifest(
        split_records,
        ontology_records=surface_ontology,
        sampling_config=sampling_config,
    )
    save_sampling_manifest(
        sampling_records,
        path=PROCESSED_DIR / "sampling_manifest.json",
    )
    with open(CKPT_MANIFEST, "wb") as f:
        pickle.dump((split_records, sampling_records, sampling_report), f)
    print(f"[5] 체크포인트 저장: {CKPT_MANIFEST}")

# ── 5 검증 ──────────────────────────────────────────────────────────
assert len(sampling_records) > 0, "[오류] 샘플링 결과가 비어있음"

train_count = sum(1 for r in sampling_records if r.split == "TL")
val_count   = sum(1 for r in sampling_records if r.split == "VL")
assert train_count > 0, "[오류] train(TL) 샘플 없음"
assert val_count   > 0, "[오류] val(VL) 샘플 없음"

# canonical_class_name이 모두 영문인지 확인
_korean_sr = [
    r.canonical_class_name for r in sampling_records
    if any(ord(c) > 127 for c in r.canonical_class_name)
]
assert len(_korean_sr) == 0, \
    f"[오류] 샘플링 레코드에 한국어 canonical_class_name: {set(_korean_sr)}"

print("\n✅ Cell 5 완료")
print(f"   선택 train (TL): {train_count:,}")
print(f"   선택 val   (VL): {val_count:,}")
print(f"   합계            : {len(sampling_records):,}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 6 — 압축 해제 + export_segmentation_dataset               ║
# ╚══════════════════════════════════════════════════════════════════╝
import shutil, zipfile
from vision.src.data import export_segmentation_dataset
from vision.src.data.normalization import NormalizedAnnotation

# ── 6-A. 기존 Korean-named curated 디렉터리 정리 ────────────────────
# canonical_class_name 버그 수정으로 이제 영문 디렉터리를 생성함.
# 이전 세션의 한국어 디렉터리가 남아있으면 제거한다.
if CURATED_ROOT.exists():
    _korean_dirs = [
        d for d in CURATED_ROOT.iterdir()
        if d.is_dir() and any(ord(c) > 127 for c in d.name)
    ]
    if _korean_dirs:
        print(f"[6-A] 이전 한국어 디렉터리 {len(_korean_dirs)}개 제거 중...")
        for d in _korean_dirs:
            shutil.rmtree(d)
            print(f"       삭제: {d.name}")

# ── 6-B. 영문 디렉터리가 이미 있으면 export 스킵 ────────────────────
CURATED_ROOT.mkdir(parents=True, exist_ok=True)
_english_dirs = [
    d for d in CURATED_ROOT.iterdir()
    if d.is_dir() and all(ord(c) <= 127 for c in d.name)
]
SKIP_EXPORT = len(_english_dirs) > 0

if SKIP_EXPORT:
    print(f"[6] export 스킵 — 영문 디렉터리 {len(_english_dirs)}개 이미 존재")
    for d in _english_dirs:
        print(f"    {d.name}")
else:
    # ── 6-C. sampled_surface.zip 압축 해제 ──────────────────────────
    SAMPLED_ZIP = DRIVE_ZIPS / "sampled_surface.zip"
    if not SAMPLED_IMAGES_DIR.exists() or not any(SAMPLED_IMAGES_DIR.iterdir()):
        print("[6-C] sampled_surface.zip 압축 해제 중...")
        assert SAMPLED_ZIP.exists(), \
            f"[오류] sampled_surface.zip 없음: {SAMPLED_ZIP}\n"\
            "Drive 경로를 확인하거나 DRIVE_ZIPS 변수를 수정하세요."
        SAMPLED_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(SAMPLED_ZIP) as zf:
            zf.extractall(SAMPLED_IMAGES_DIR)
        _img_count = sum(1 for _ in SAMPLED_IMAGES_DIR.rglob("*") if _.is_file())
        print(f"[6-C] 추출 완료: {_img_count:,}개 파일")
    else:
        _img_count = sum(1 for _ in SAMPLED_IMAGES_DIR.rglob("*") if _.is_file())
        print(f"[6-C] 이미 추출됨: {_img_count:,}개 파일")

    # labels.zip (있을 경우에만 추출)
    LABELS_ZIP = DRIVE_ZIPS / "labels.zip"
    if LABELS_ZIP.exists():
        if not LABELS_FLAT_DIR.exists() or not any(LABELS_FLAT_DIR.rglob("*.json")):
            print("[6-C] labels.zip 압축 해제 중...")
            LABELS_FLAT_DIR.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(LABELS_ZIP) as zf:
                zf.extractall(LABELS_FLAT_DIR)
            print("[6-C] labels.zip 추출 완료")
    else:
        print("[6-C] labels.zip 없음 — 라벨이 이미지와 함께 있거나 Drive에서 직접 읽습니다.")
        LABELS_FLAT_DIR = SAMPLED_IMAGES_DIR   # fallback: 이미지 디렉터리에 json이 있을 경우

    # ── 6-D. export 실행 ────────────────────────────────────────────
    print("[6-D] export_segmentation_dataset 실행 중...")
    export_result = export_segmentation_dataset(
        (r for r in normalized if r.annotation_domain == TARGET_DOMAIN),
        ontology_records=surface_ontology,
        resized_root=SAMPLED_IMAGES_DIR,
        labels_root=LABELS_FLAT_DIR,
        output_root=CURATED_ROOT,
    )
    print(f"[6-D] export 완료")
    print(f"    exported : {export_result.exported_count:,}")
    print(f"    blocked  : {export_result.blocked_count:,}")
    print(f"    skipped  : {export_result.skipped_count:,}")

# ── 6 검증 ──────────────────────────────────────────────────────────
from collections import defaultdict

_class_dirs = [d for d in CURATED_ROOT.iterdir() if d.is_dir()]
assert len(_class_dirs) > 0, \
    f"[오류] CURATED_ROOT가 비어있음: {CURATED_ROOT}"

_stats = defaultdict(lambda: defaultdict(int))
for class_dir in sorted(_class_dirs):
    for split in ("train", "val"):
        img_dir = class_dir / "images" / split
        lbl_dir = class_dir / "labels" / split
        _stats[class_dir.name][split] = (
            sum(1 for _ in img_dir.glob("*")) if img_dir.exists() else 0
        )

assert all(
    not any(ord(c) > 127 for c in d.name) for d in _class_dirs
), "[오류] 한국어 클래스 디렉터리가 남아있음 — 위 6-A 블록을 확인하세요."

print("\n✅ Cell 6 완료")
print(f"   CURATED_ROOT: {CURATED_ROOT}")
for cls_name, splits in sorted(_stats.items()):
    print(f"   {cls_name:40s} train={splits['train']:5d}  val={splits['val']:5d}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 7 — 학습 전 데이터셋 검증                                  ║
# ╚══════════════════════════════════════════════════════════════════╝
from vision.src.settings import build_segmentation_runtime_config
from vision.src.training import build_training_artifacts, write_yolo_dataset_yaml

RUN_NAME = "siren-seg-v1"

# ── 7-A. runtime config 빌드 (label map 기반 class_names) ───────────
runtime = build_segmentation_runtime_config(surface_ontology)

assert len(runtime.class_names) > 0, \
    "[오류] runtime.class_names가 비어있음\n"\
    "→ surface_ontology의 support_bucket 분포를 확인하세요 (모두 'review'일 수 있음)."

_korean_class = [n for n in runtime.class_names if any(ord(c) > 127 for c in n)]
assert len(_korean_class) == 0, \
    f"[오류] runtime.class_names에 한국어 포함: {_korean_class}\n"\
    "→ Cell 4 체크포인트({CKPT_ONTOLOGY}) 삭제 후 재실행하세요."

print(f"[7-A] runtime.class_names ({len(runtime.class_names)}개):")
for i, name in enumerate(runtime.class_names):
    print(f"    [{i}] {name}")

# ── 7-B. curated dirs가 runtime class_names와 매칭되는지 확인 ────────
_existing = {d.name for d in CURATED_ROOT.iterdir() if d.is_dir()}
_missing  = [n for n in runtime.class_names
             if not (CURATED_ROOT / n / "images" / "train").exists()
             or not (CURATED_ROOT / n / "images" / "val").exists()]

if _missing:
    print(f"[경고] train/val 디렉터리 없는 클래스 {len(_missing)}개:")
    for n in _missing:
        print(f"    {n}")
    print("    → support_bucket='review' 또는 export에서 skip된 클래스일 수 있음")
    print("    → 학습에서 해당 클래스는 제외됩니다")

_trainable = [n for n in runtime.class_names
              if (CURATED_ROOT / n / "images" / "train").exists()
              and (CURATED_ROOT / n / "images" / "val").exists()]
assert len(_trainable) > 0, \
    "[오류] 학습 가능한 클래스가 없음 — Cell 6 export를 재실행하세요."

# ── 7-C. YOLO dataset.yaml 미리 생성 및 검증 ────────────────────────
from vision.src.settings import VisionRuntimeConfig, VisionPaths
from dataclasses import replace

# runtime에 curated_root 반영
_paths = replace(runtime.paths, resized_root=CURATED_ROOT, drive_runs_root=DRIVE_RUNS)
runtime = replace(runtime, paths=_paths, device="cuda" if __import__('torch').cuda.is_available() else "cpu")

artifacts = build_training_artifacts(
    runtime, run_name=RUN_NAME, curated_root=CURATED_ROOT
)
write_yolo_dataset_yaml(artifacts)

_yaml_text = artifacts.data_yaml_path.read_text()
assert "train:" in _yaml_text and "val:" in _yaml_text, \
    "[오류] dataset.yaml에 train/val 항목 없음"
assert all(name in _yaml_text for name in runtime.class_names), \
    "[오류] dataset.yaml에 class_names 누락"

print(f"\n[7-C] dataset.yaml 생성: {artifacts.data_yaml_path}")
print(_yaml_text)

print("\n✅ Cell 7 완료")
print(f"   학습 가능 클래스: {len(_trainable)}개")
print(f"   device: {runtime.device}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 8 — YOLOv8n-seg 학습                                       ║
# ╚══════════════════════════════════════════════════════════════════╝
# Cell 7에서 runtime, artifacts, RUN_NAME, CURATED_ROOT가 설정되어 있어야 함.

import time
from vision.src.training import train_yolo_segmentation

print(f"[8] 학습 시작: run_name={RUN_NAME!r}")
print(f"    class_names : {runtime.class_names}")
print(f"    epochs      : {runtime.epochs}")
print(f"    batch_size  : {runtime.batch_size}")
print(f"    image_size  : {runtime.image_size}")
print(f"    device      : {runtime.device}")
print(f"    curated_root: {CURATED_ROOT}")

_t0 = time.time()

train_result = train_yolo_segmentation(
    runtime=runtime,
    run_name=RUN_NAME,
    curated_root=CURATED_ROOT,
    ontology_records=surface_ontology,
)

_elapsed = time.time() - _t0

# ── 8 검증 ──────────────────────────────────────────────────────────
assert train_result.best_weight_path is not None, \
    "[오류] best.pt 경로가 None — 학습이 정상 완료되지 않았습니다."
assert train_result.best_weight_path.exists(), \
    f"[오류] best.pt 파일 없음: {train_result.best_weight_path}"

_file_size_mb = train_result.best_weight_path.stat().st_size / 1e6
assert _file_size_mb > 1.0, \
    f"[오류] best.pt 파일 크기가 너무 작음: {_file_size_mb:.1f} MB (손상 의심)"

# Drive 미러 확인
if train_result.drive_best_weight_path and train_result.drive_best_weight_path.exists():
    print(f"[8] Drive 미러 확인: {train_result.drive_best_weight_path}")
else:
    print(f"[경고] Drive 미러 없음 — 수동으로 복사하세요:")
    print(f"    shutil.copy('{train_result.best_weight_path}', '<Drive 경로>')")

# 최종 메트릭 출력 (YOLO train_result 객체 구조에 따라 다를 수 있음)
_metrics_path = train_result.artifacts.local_run_dir / "results.csv"
if _metrics_path.exists():
    import csv
    with open(_metrics_path) as f:
        rows = list(csv.DictReader(f))
    if rows:
        _last = rows[-1]
        _map50 = _last.get("metrics/mAP50(B)", _last.get("metrics/mAP50-95(B)", "N/A"))
        print(f"[8] 최종 mAP50: {_map50}")

print(f"\n✅ Cell 8 완료")
print(f"   경과 시간    : {_elapsed/60:.1f}분")
print(f"   best.pt      : {train_result.best_weight_path}")
print(f"   best.pt 크기 : {_file_size_mb:.1f} MB")